# 04 — Fine-tuning PhoBERT cho Phân tích Cảm xúc

Notebook này thực hiện fine-tune **PhoBERT** (`vinai/phobert-base-v2`) cho bài toán
**Sentiment Classification** (Positive / Neutral / Negative) từ dữ liệu đánh giá sản phẩm tiếng Việt.

```
Load processed data → Dataset/DataLoader → Fine-tune PhoBERT
    → Evaluate → Save HuggingFace format → (Optional) Push to Hub
```

**Input:** `data/processed/*_labeled.csv` + `class_weights.json`  
**Output:** `models/phobert_sentiment/` (HuggingFace format, sẵn sàng deploy)


## 1. Khởi tạo môi trường

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install transformers datasets accelerate scikit-learn seaborn -q


In [3]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix,
)
warnings.filterwarnings('ignore')

# ── Cấu hình đường dẫn ────────────────────────────────────────
# ⚠️  Thay đổi BASE_PATH cho phù hợp với Drive của bạn
BASE_PATH  = '/content/drive/MyDrive/sentiment_analyst/'
DATA_PATH  = BASE_PATH + 'data/processed/'
MODEL_PATH = BASE_PATH + 'models/phobert_sentiment/'
CKPT_PATH  = BASE_PATH + 'outputs/checkpoints/'
LOG_PATH   = BASE_PATH + 'outputs/logs/'
FIG_PATH   = BASE_PATH + 'outputs/figures/'

for p in [DATA_PATH, MODEL_PATH, CKPT_PATH, LOG_PATH, FIG_PATH]:
    os.makedirs(p, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False


Device: cuda
GPU: Tesla T4


## 2. Siêu tham số

In [4]:
# ── Model ────────────────────────────────────────────────────
PHOBERT_NAME = 'vinai/phobert-base-v2'
NUM_LABELS   = 3           # Negative=0, Neutral=1, Positive=2
MAX_LEN      = 128

# ── Training ─────────────────────────────────────────────────
BATCH_SIZE       = 32
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 10
WARMUP_RATIO     = 0.06    # ~146 steps — phù hợp với dataset nhỏ
MAX_GRAD_NORM    = 1.0     # gradient clipping
EARLY_STOP_PAT   = 4       # an toàn hơn với PhoBERT
DROPOUT_RATE     = 0.1

USE_AMP          = True
ACCUM_STEPS      = 2
LLRD_FACTOR      = 0.95
USE_FOCAL_LOSS   = True
K_FOLDS          = 5

# ── Label mapping ─────────────────────────────────────────────
# ⚠️  Phải khớp với LABEL_ENCODE trong 01_Data_Preprocessing.ipynb
# weights_list[0]=Negative, [1]=Neutral, [2]=Positive
LABEL_ENCODE = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
LABEL_DECODE = {v: k for k, v in LABEL_ENCODE.items()}
ID2LABEL     = LABEL_DECODE
LABEL2ID     = LABEL_ENCODE

print('Cấu hình:')
for k, v in {
    'Model': PHOBERT_NAME, 'MAX_LEN': MAX_LEN, 'BATCH_SIZE': BATCH_SIZE,
    'LR': LEARNING_RATE, 'EPOCHS': NUM_EPOCHS, 'EARLY_STOP': EARLY_STOP_PAT
}.items():
    print(f'  {k:15s}: {v}')


Cấu hình:
  Model          : vinai/phobert-base-v2
  MAX_LEN        : 128
  BATCH_SIZE     : 32
  LR             : 2e-05
  EPOCHS         : 10
  EARLY_STOP     : 4


## 3. Tải dữ liệu

In [5]:
train_df = pd.read_csv(DATA_PATH + 'train_labeled.csv')
val_df   = pd.read_csv(DATA_PATH + 'val_labeled.csv')
test_df  = pd.read_csv(DATA_PATH + 'test_labeled.csv')

# ✅ Bảo vệ an toàn: tự map nhãn nếu cột 'label' chưa tồn tại
for df in [train_df, val_df, test_df]:
    if 'label' not in df.columns:
        df['label'] = df['sentiment'].map(LABEL_ENCODE)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Các cột: {list(train_df.columns)}')
train_df[['Review', 'sentiment', 'label']].head(3)


Train: 7,760 | Val: 845 | Test: 2,146
Các cột: ['Review', 'sentiment', 'label']


,Review,sentiment,label
0,giày đẹp đi êm lắm,Positive,2
1,mình săn sale với giá khá rẻ chất_lượng ok shi...,Positive,2
2,hình_ảnh và video chỉ mang tính_chất minh_họa ...,Positive,2


In [6]:
# Load class weights từ file JSON
with open(DATA_PATH + 'class_weights.json', encoding='utf-8') as f:
    cw_data = json.load(f)

class_weights = torch.tensor(
    cw_data['weights_list'], dtype=torch.float
).to(device)  # [w_Negative, w_Neutral, w_Positive]

print('Class weights:', cw_data['weights_list'])
print('Label map    :', cw_data['label_encode'])


Class weights: [3.2742616033755274, 1.7406908927770302, 0.47167517627036226]
Label map    : {'Negative': 0, 'Neutral': 1, 'Positive': 2}


## 4. Dataset & DataLoader

In [7]:
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_NAME)

class PhoBERTDataset(Dataset):
    """
    Dataset cho PhoBERT Sentiment Classification.
    """
    def __init__(self, df, tokenizer, max_len):
        self.reviews   = df['Review'].astype(str).tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.reviews[idx],
            max_length      = self.max_len,
            padding         = 'max_length',
            truncation      = True,
            return_tensors  = 'pt',
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long),
        }

# K-Fold: Gộp train và val
full_train_df = pd.concat([train_df, val_df]).reset_index(drop=True)
full_train_dataset = PhoBERTDataset(full_train_df, tokenizer, MAX_LEN)
test_dataset  = PhoBERTDataset(test_df,  tokenizer, MAX_LEN)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=(device.type == 'cuda'))

print(f'Full Train size: {len(full_train_dataset)} | Test: {len(test_loader)}')


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Full Train size: 8605 | Test: 68


## 5. Các hàm hỗ trợ

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma  = gamma

    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

def get_llrd_params(model, base_lr, factor):
    params = []
    params.append({'params': model.classifier.parameters(), 'lr': base_lr})
    for i, layer in enumerate(reversed(model.roberta.encoder.layer)):
        lr = base_lr * (factor ** (i + 1))
        params.append({'params': layer.parameters(), 'lr': lr})
    params.append({'params': model.roberta.embeddings.parameters(), 'lr': base_lr * (factor ** 13)})
    return params

def train_epoch(model, loader, optimizer, scheduler, criterion, device, scaler):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        with autocast(enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS
        preds = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss    = criterion(outputs.logits, labels)

            total_loss += loss.item()
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, acc, f1, all_preds, all_labels


## 6. Huấn luyện với K-Fold Cross Validation

Áp dụng Stratified K-Fold, LLRD, AMP và Focal Loss.


In [ ]:
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(full_train_df, full_train_df['label'])):
    print(f'\n' + '='*50)
    print(f'🚀 FOLD {fold + 1}/{K_FOLDS}')
    print('='*50)

    train_sub = Subset(full_train_dataset, train_idx)
    val_sub   = Subset(full_train_dataset, val_idx)

    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, pin_memory=(device.type=='cuda'))
    val_loader   = DataLoader(val_sub,   batch_size=BATCH_SIZE, shuffle=False, pin_memory=(device.type=='cuda'))

    model = AutoModelForSequenceClassification.from_pretrained(
        PHOBERT_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID
    )
    model.config.hidden_dropout_prob = DROPOUT_RATE
    model.config.attention_probs_dropout_prob = DROPOUT_RATE
    model = model.to(device)

    optimizer_grouped_parameters = get_llrd_params(model, LEARNING_RATE, LLRD_FACTOR)
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, weight_decay=0.01)

    total_steps  = len(train_loader) * NUM_EPOCHS // ACCUM_STEPS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    if USE_FOCAL_LOSS:
        criterion = FocalLoss(weight=class_weights, gamma=2.0)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights)

    scaler = GradScaler(enabled=USE_AMP)

    best_val_f1 = 0.0
    best_epoch = 0
    early_stop_cnt = 0
    history = {'train_loss':[], 'train_f1':[], 'val_loss':[], 'val_f1':[], 'lr':[]}

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, scaler)
        vl_loss, vl_acc, vl_f1, _, _ = evaluate(model, val_loader, criterion, device)

        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(tr_loss)
        history['train_f1'].append(tr_f1)
        history['val_loss'].append(vl_loss)
        history['val_f1'].append(vl_f1)
        history['lr'].append(current_lr)

        note = ''
        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_epoch = epoch
            early_stop_cnt = 0
            note = '✅ best'
            model.save_pretrained(CKPT_PATH + f'best_model_fold_{fold+1}')
            tokenizer.save_pretrained(CKPT_PATH + f'best_model_fold_{fold+1}')
        else:
            early_stop_cnt += 1
            note = f'EStop {early_stop_cnt}/{EARLY_STOP_PAT}'

        print(f'Epoch {epoch:>2} | LR={current_lr:.2e} | TrL={tr_loss:.4f} TrF1={tr_f1:.4f} | VlL={vl_loss:.4f} VlF1={vl_f1:.4f} | {note}')

        if early_stop_cnt >= EARLY_STOP_PAT:
            print(f'⏹ Early stopping Fold {fold+1} tại epoch {epoch}.')
            break

    print(f'✅ Fold {fold+1} Best Val F1: {best_val_f1:.4f}')
    fold_results.append(best_val_f1)

    with open(LOG_PATH + f'training_history_fold_{fold+1}.json', 'w', encoding='utf-8') as fout:
        json.dump(history, fout, indent=2)

print(f'\n🏆 K-FOLD AVERAGE BEST VAL F1: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}')



🚀 FOLD 1/5


pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

...

Epoch  3 | LR=1.49e-05 | TrL=0.1994 TrF1=0.7316 | VlL=0.2615 VlF1=0.7211 | ✅ best
Epoch  4 | LR=1.28e-05 | TrL=0.1610 TrF1=0.7502 | VlL=0.2769 VlF1=0.7038 | EStop 1/4


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch  5 | LR=1.06e-05 | TrL=0.1341 TrF1=0.7743 | VlL=0.3683 VlF1=0.7383 | ✅ best
Epoch  6 | LR=8.50e-06 | TrL=0.1076 TrF1=0.7944 | VlL=0.3690 VlF1=0.7316 | EStop 1/4


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch  7 | LR=6.38e-06 | TrL=0.0891 TrF1=0.8135 | VlL=0.4403 VlF1=0.7413 | ✅ best
Epoch  8 | LR=4.25e-06 | TrL=0.0780 TrF1=0.8320 | VlL=0.4126 VlF1=0.7405 | EStop 1/4


## 7. Biểu đồ quá trình huấn luyện (Best Fold)

In [ ]:
# Load history của fold tốt nhất để vẽ
best_fold = np.argmax(fold_results) + 1
with open(LOG_PATH + f'training_history_fold_{best_fold}.json', 'r', encoding='utf-8') as f:
    history = json.load(f)
epochs_range = range(1, len(history['train_loss']) + 1)
best_epoch = np.argmax(history['val_f1']) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2196F3')
axes[0].plot(epochs_range, history['val_loss'],   's-', label='Val Loss',   color='#F44336')
axes[0].axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
axes[0].set_title(f'Loss (Fold {best_fold})', fontweight='bold', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# F1
axes[1].plot(epochs_range, history['train_f1'], 'o-', label='Train F1', color='#4CAF50')
axes[1].plot(epochs_range, history['val_f1'],   's-', label='Val F1',   color='#FF9800')
axes[1].axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
axes[1].set_title(f'F1 Macro (Fold {best_fold})', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle(f'PhoBERT Fine-tuning — Training Curves (Best Fold: {best_fold})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_PATH + 'training_curves.png', bbox_inches='tight', dpi=150)
plt.show()


## 8. Đánh giá trên tập Test

In [ ]:
# CHỌN FOLD TỐT NHẤT HOẶC ENSEMBLE
# Ở đây ta lấy fold có kết quả tốt nhất trên Validation để đánh giá Test
best_fold = np.argmax(fold_results) + 1
print(f'Dùng mô hình của Fold {best_fold} để đánh giá trên Test Set.')

best_model = AutoModelForSequenceClassification.from_pretrained(
    CKPT_PATH + f'best_model_fold_{best_fold}',
    id2label  = ID2LABEL,
    label2id  = LABEL2ID,
).to(device)

with open(DATA_PATH + 'class_weights.json', encoding='utf-8') as _f:
    _cw = json.load(_f)

if USE_FOCAL_LOSS:
    criterion = FocalLoss(weight=torch.tensor(_cw['weights_list'], dtype=torch.float).to(device), gamma=2.0)
else:
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(_cw['weights_list'], dtype=torch.float).to(device))

_, _, test_f1, test_preds, test_labels = evaluate(
    best_model, test_loader, criterion, device)

label_names = ['Negative', 'Neutral', 'Positive']
assert [LABEL_DECODE[i] for i in range(NUM_LABELS)] == label_names, \
    'label_names không khớp LABEL_DECODE!'

print('=== KẾT QUẢ TRÊN TẬP TEST ===')
print(f'Accuracy        : {accuracy_score(test_labels, test_preds):.4f}')
print(f'F1 Macro        : {f1_score(test_labels, test_preds, average="macro", zero_division=0):.4f}')
print(f'F1 Weighted     : {f1_score(test_labels, test_preds, average="weighted", zero_division=0):.4f}')
print()
print(classification_report(test_labels, test_preds,
                             target_names=label_names, zero_division=0))


In [ ]:
# Confusion Matrix
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalize theo hàng (actual)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Confusion Matrix (số lượng)'),
    (axes[1], cm_norm, '.2f', 'Confusion Matrix (% theo nhãn thực / Recall)'),
]:
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle(f'PhoBERT — Confusion Matrix trên Test set (Fold {best_fold})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_PATH + 'confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
from sklearn.metrics import classification_report as _cr

report_dict = _cr(test_labels, test_preds,
                  target_names=label_names, output_dict=True, zero_division=0)

test_results = {
    'accuracy'   : float(accuracy_score(test_labels, test_preds)),
    'f1_macro'   : float(f1_score(test_labels, test_preds, average='macro',    zero_division=0)),
    'f1_weighted': float(f1_score(test_labels, test_preds, average='weighted', zero_division=0)),
    'precision_macro': float(precision_score(test_labels, test_preds, average='macro', zero_division=0)),
    'recall_macro'   : float(recall_score(test_labels, test_preds, average='macro', zero_division=0)),
    'best_epoch' : best_epoch,
    'best_val_f1': float(best_val_f1),
    'per_class'  : report_dict,
}
with open(LOG_PATH + 'test_results.json', 'w', encoding='utf-8') as f:
    json.dump(test_results, f, indent=2, ensure_ascii=False)

print('✅ Đã lưu kết quả vào test_results.json')
print(f'  Accuracy    : {test_results["accuracy"]:.4f}')
print(f'  F1 Macro    : {test_results["f1_macro"]:.4f}')
print(f'  F1 Weighted : {test_results["f1_weighted"]:.4f}')
print()
print('Per-class:')
for cls in label_names:
    r = report_dict[cls]
    print(f'  {cls:10s} | P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1-score"]:.4f}  N={int(r["support"])}')


## 9. Lưu mô hình (Best Fold) để Deploy


In [ ]:
model_card = """
---
language: vi
tags:
  - sentiment-analysis
  - vietnamese
  - phobert
  - text-classification
datasets:
  - custom-shopee-reviews
metrics:
  - f1
  - accuracy
pipeline_tag: text-classification
---

# PhoBERT Vietnamese Sentiment Analysis

Mô hình phân tích cảm xúc tiếng Việt fine-tuned từ [vinai/phobert-base-v2](https://huggingface.co/vinai/phobert-base-v2)
trên tập dữ liệu đánh giá sản phẩm thương mại điện tử (Shopee).

## Nhãn
| Label | ID | Mô tả |
|-------|----|-----------|
| Negative | 0 | Đánh giá tiêu cực |
| Neutral  | 1 | Đánh giá trung tính / hỗn hợp |
| Positive | 2 | Đánh giá tích cực |

## Tiền xử lý bắt buộc
Văn bản **phải được tách từ** bằng `underthesea.word_tokenize(text, format='text')`
trước khi đưa vào model.

## Kết quả
| Metric | Score |
|--------|-------|
| Accuracy | - |
| F1 Macro | - |
| F1 Weighted | - |
"""

best_model.save_pretrained(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

with open(MODEL_PATH + 'README.md', 'w', encoding='utf-8') as f:
    f.write(model_card)

with open(MODEL_PATH + 'README.md', 'r', encoding='utf-8') as _rf:
    readme_content = _rf.read()
readme_content = readme_content.replace(
    '| Accuracy | - |',
    f'| Accuracy | {test_results["accuracy"]:.4f} |'
).replace(
    '| F1 Macro | - |',
    f'| F1 Macro | {test_results["f1_macro"]:.4f} |'
).replace(
    '| F1 Weighted | - |',
    f'| F1 Weighted | {test_results["f1_weighted"]:.4f} |'
)
with open(MODEL_PATH + 'README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)

print('✅ Model đã lưu tại:', MODEL_PATH)


## 10. Demo Inference

In [ ]:
from underthesea import word_tokenize
import unicodedata, re

TEENCODE = {
    'ko':'không','k':'không','kh':'không','đc':'được','dc':'được',
    'cx':'cũng','nma':'nhưng mà','sp':'sản phẩm','mn':'mọi người',
    'mng':'mọi người','vl':'rất','vkl':'rất','oke':'ok','okie':'ok',
    'oki':'ok','okela':'ok','okila':'ok',
    'ship':'giao hàng','ib':'nhắn_tin','rep':'phản_hồi','feedback':'đánh_giá',
}
EMOJI = {
    '❤️':'tích_cực','🧡':'tích_cực','💛':'tích_cực','💚':'tích_cực',
    '💙':'tích_cực','💜':'tích_cực','🤍':'tích_cực','❣️':'tích_cực',
    '💗':'tích_cực','💓':'tích_cực','😍':'tích_cực','🥰':'tích_cực',
    '😊':'tích_cực','😄':'tích_cực','😁':'tích_cực','🤩':'tích_cực',
    '😻':'tích_cực','🌟':'tích_cực','⭐':'tích_cực','👍':'tích_cực',
    '💪':'tích_cực','✅':'tích_cực','👌':'tích_cực',
    '😡':'tiêu_cực','😠':'tiêu_cực','🤬':'tiêu_cực','😤':'tiêu_cực',
    '👎':'tiêu_cực','❌':'tiêu_cực','😭':'tiêu_cực','😢':'tiêu_cực','🥲':'tiêu_cực',
}

def preprocess(text):
    text = text.lower()
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    for emo, val in EMOJI.items(): text = text.replace(emo, f' {val} ')
    words = [TEENCODE.get(w, w) for w in text.split()]
    text  = ' '.join(words)
    text  = re.sub(r'!+', '!', text)
    text  = re.sub(r'\?+', '?', text)
    text  = re.sub(r'[^\w\s!?_]', ' ', text)
    text  = re.sub(r'\s+', ' ', text).strip()
    if text: text = word_tokenize(text, format='text')
    return text

try:
    best_model
except NameError:
    print('⏳ best_model chưa được định nghĩa. Đang load từ checkpoint...')
    best_model = AutoModelForSequenceClassification.from_pretrained(
        CKPT_PATH + f'best_model_fold_1',
        id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(device)
    print('✅ Đã load best_model (mặc định fold 1).')

def predict(text, model, tokenizer, device, threshold=0.5):
    model.eval()
    processed = preprocess(text)
    enc = tokenizer(processed, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        outputs = model(
            input_ids      = enc['input_ids'].to(device),
            attention_mask = enc['attention_mask'].to(device),
        )
    probs      = torch.softmax(outputs.logits, dim=-1).cpu().squeeze()
    pred_id    = probs.argmax().item()
    confidence = probs[pred_id].item()
    if confidence < threshold:
        return {'label': 'Unknown', 'confidence': confidence,
                'probs': {ID2LABEL[i]: round(p.item(), 4) for i, p in enumerate(probs)},
                'preprocessed': processed}
    return {
        'label'       : ID2LABEL[pred_id],
        'confidence'  : confidence,
        'probs'       : {ID2LABEL[i]: round(p.item(), 4) for i, p in enumerate(probs)},
        'preprocessed': processed,
    }

test_cases = [
    'Giao hàng nhanh 👍, sản phẩm đẹp lắm mọi người nên mua!',
    'Chất lượng bình thường, giá hơi cao nhưng ship nhanh',
    'Hàng giả, chất lượng kém, giao hàng chậm. Không mua lại!',
    'shop nhiệt tình ko ngờ đẹp vl luôn 😍😍',
]
print('=== DEMO INFERENCE ===\n')
for text in test_cases:
    result = predict(text, best_model, tokenizer, device)
    print(f'Input      : {text}')
    print(f'Preprocessed: {result["preprocessed"][:80]}')
    print(f'Kết quả    : {result["label"]} (confidence: {result["confidence"]:.4f})')
    print(f'Xác suất   : {result["probs"]}')
    print()
